In [1]:
# Cell 1 — Install dependencies
!pip install -q transformers datasets peft accelerate huggingface_hub
!pip install -q torchvision Pillow scikit-learn
print('✅ All packages installed!')

✅ All packages installed!


In [2]:
# Cell 2 — Verify GPU
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠️  No GPU — go to Runtime > Change runtime type > T4 GPU')

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [3]:
# Cell 3 — Login to HuggingFace Hub
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print('✅ Logged in to HuggingFace Hub!')
except Exception:
    print('⚠️  HF_TOKEN not found in Secrets — adding login prompt')
    login()

✅ Logged in to HuggingFace Hub!


In [7]:
# Cell 4 — FER2013 Kaggle se load karo
!pip install -q kaggle

# kaggle.json upload karo
from google.colab import files
files.upload()  # apna kaggle.json upload karo

import os
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('/content/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 600)

# Dataset download karo
!kaggle datasets download -d msambare/fer2013
!unzip -q fer2013.zip

print('✅ FER2013 downloaded!')
print('Files:', os.listdir('/content'))

Saving kaggle.json to kaggle (1).json
Dataset URL: https://www.kaggle.com/datasets/msambare/fer2013
License(s): DbCL-1.0
  0% 0.00/60.3M [00:00<?, ?B/s]
100% 60.3M/60.3M [00:00<00:00, 1.77GB/s]
✅ FER2013 downloaded!
Files: ['.config', 'kaggle (1).json', 'test', 'train', 'fer2013.zip', 'sample_data']


In [8]:
# Cell 5 — Explore dataset structure
import os

train_dir = '/content/train'
test_dir  = '/content/test'

emotions = os.listdir(train_dir)
print('Emotions found:', emotions)
print(f'\nTotal emotion classes: {len(emotions)}')
print('\nSamples per emotion (train):')
for emotion in sorted(emotions):
    count = len(os.listdir(f'{train_dir}/{emotion}'))
    print(f'  {emotion}: {count} images')

total_train = sum(len(os.listdir(f'{train_dir}/{e}')) for e in emotions)
total_test  = sum(len(os.listdir(f'{test_dir}/{e}'))  for e in os.listdir(test_dir))
print(f'\nTotal train: {total_train}')
print(f'Total test:  {total_test}')

Emotions found: ['angry', 'disgust', 'surprise', 'fear', 'neutral', 'happy', 'sad']

Total emotion classes: 7

Samples per emotion (train):
  angry: 3995 images
  disgust: 436 images
  fear: 4097 images
  happy: 7215 images
  neutral: 4965 images
  sad: 4830 images
  surprise: 3171 images

Total train: 28709
Total test:  7178


In [9]:
# Cell 6 — Load and preprocess images
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Image transformations
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),        # ViT needs 224x224
    transforms.Grayscale(num_output_channels=3),  # FER is grayscale, ViT needs 3 channels
    transforms.RandomHorizontalFlip(),    # Data augmentation
    transforms.RandomRotation(10),        # Slight rotation augmentation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Load datasets
train_dataset = datasets.ImageFolder('/content/train', transform=train_transforms)
test_dataset  = datasets.ImageFolder('/content/test',  transform=test_transforms)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2)

print(f'✅ Data loaded!')
print(f'Classes: {train_dataset.classes}')
print(f'Train batches: {len(train_loader)}')
print(f'Test batches:  {len(test_loader)}')

# Show one batch shape
images, labels = next(iter(train_loader))
print(f'Batch shape: {images.shape}')
print(f'Labels shape: {labels.shape}')

✅ Data loaded!
Classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Train batches: 898
Test batches:  225
Batch shape: torch.Size([32, 3, 224, 224])
Labels shape: torch.Size([32])


In [11]:
# Cell 7 — Load ViT model (fixed)
from transformers import ViTForImageClassification, AutoImageProcessor
import torch

MODEL_NAME = 'google/vit-base-patch16-224'
NUM_CLASSES = 7
EMOTIONS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

print(f'Loading {MODEL_NAME}...')

model = ViTForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    id2label={i: e for i, e in enumerate(EMOTIONS)},
    label2id={e: i for i, e in enumerate(EMOTIONS)},
    ignore_mismatched_sizes=True
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'✅ ViT loaded!')
print(f'Total parameters: {total_params:.1f}M')
print(f'Device: {device}')

Loading google/vit-base-patch16-224...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([7])          
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([7, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


✅ ViT loaded!
Total parameters: 85.8M
Device: cuda


In [13]:
# Cell 8 — Configure LoRA for ViT (fixed)
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=['query', 'value'],
    bias='none'
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'✅ LoRA applied!')
print(f'Trainable parameters: {trainable:,} ({100 * trainable / total:.2f}% of total)')
print(f'Frozen parameters:    {total - trainable:,}')

✅ LoRA applied!
Trainable parameters: 589,824 (0.68% of total)
Frozen parameters:    85,804,039


In [14]:
# Cell 9 — Training setup
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Loss function — weighted because disgust class has very few samples
class_counts = [3995, 436, 4097, 7215, 4965, 4830, 3171]
total = sum(class_counts)
weights = torch.tensor([total/c for c in class_counts]).to(device)
weights = weights / weights.sum() * len(class_counts)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=2e-4,
    weight_decay=0.01
)
scheduler = CosineAnnealingLR(optimizer, T_max=10)

print('✅ Training setup ready!')
print(f'Class weights: {[round(w.item(), 2) for w in weights]}')
print(f'Optimizer: AdamW lr=2e-4')
print(f'Scheduler: CosineAnnealingLR')

✅ Training setup ready!
Class weights: [0.48, 4.4, 0.47, 0.27, 0.39, 0.4, 0.6]
Optimizer: AdamW lr=2e-4
Scheduler: CosineAnnealingLR


In [15]:
# Cell 10 — Train the model!
from tqdm import tqdm

EPOCHS = 10

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in tqdm(loader, desc='Training'):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(pixel_values=images)
        loss = criterion(outputs.logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), correct / total

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Evaluating'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(pixel_values=images)
            loss = criterion(outputs.logits, labels)
            total_loss += loss.item()
            correct += (outputs.logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total

print('🚀 Training started!')
best_acc = 0
history = []

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc     = eval_epoch(model, test_loader, criterion)
    scheduler.step()

    history.append({
        'epoch': epoch+1,
        'train_loss': train_loss, 'train_acc': train_acc,
        'val_loss': val_loss,     'val_acc': val_acc
    })

    print(f'Epoch {epoch+1}/{EPOCHS} — Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}')

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pt')
        print(f'  ✅ Best model saved! Val acc: {best_acc:.4f}')

print(f'\n🎉 Training complete! Best val accuracy: {best_acc:.4f}')

🚀 Training started!


Evaluating: 100%|██████████| 225/225 [01:32<00:00,  2.43it/s]


Epoch 1/10 — Train Loss: 1.3446 Acc: 0.5045 | Val Loss: 1.1353 Acc: 0.5669
  ✅ Best model saved! Val acc: 0.5669


Evaluating: 100%|██████████| 225/225 [01:32<00:00,  2.43it/s]


Epoch 2/10 — Train Loss: 1.0582 Acc: 0.6028 | Val Loss: 1.0501 Acc: 0.6045
  ✅ Best model saved! Val acc: 0.6045


...

Epoch 8/10 — Train Loss: 0.7408 Acc: 0.7104 | Val Loss: 0.9209 Acc: 0.6552
  ✅ Best model saved! Val acc: 0.6552


Evaluating: 100%|██████████| 225/225 [01:32<00:00,  2.43it/s]


Epoch 9/10 — Train Loss: 0.7266 Acc: 0.7166 | Val Loss: 0.9263 Acc: 0.6555
  ✅ Best model saved! Val acc: 0.6555


Evaluating: 100%|██████████| 225/225 [01:32<00:00,  2.43it/s]


Epoch 10/10 — Train Loss: 0.7127 Acc: 0.7201 | Val Loss: 0.9178 Acc: 0.6574
  ✅ Best model saved! Val acc: 0.6574

🎉 Training complete! Best val accuracy: 0.6574


In [17]:
# Emergency save — foran run karo training complete hone pe
from google.colab import drive
import torch, os

drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/emotion_checkpoints', exist_ok=True)

# Best model save
torch.save(model.state_dict(),
           '/content/drive/MyDrive/emotion_checkpoints/best_model_final.pt')

# Full checkpoint save
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'best_acc': best_acc,
    'history': history,
    'emotions': ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
}, '/content/drive/MyDrive/emotion_checkpoints/full_checkpoint.pt')

print(f'✅ Model saved to Drive!')
print(f'Best accuracy: {best_acc:.4f}')
print(f'Files saved:')
print(os.listdir('/content/drive/MyDrive/emotion_checkpoints'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Model saved to Drive!
Best accuracy: 0.6574
Files saved:
['best_model_final.pt', 'full_checkpoint.pt']


In [18]:
# Cell 12 — Push model to HuggingFace Hub
from huggingface_hub import HfApi
import torch

HF_USERNAME = 'kashanikram'
MODEL_SAVE_NAME = f'{HF_USERNAME}/facial-emotion-vit'

print('Saving model...')
# Merge LoRA weights before saving
merged_model = model.merge_and_unload()

merged_model.save_pretrained('./emotion-model-final')
print('Pushing to Hub...')
merged_model.push_to_hub(MODEL_SAVE_NAME)

from transformers import AutoImageProcessor
processor = AutoImageProcessor.from_pretrained('google/vit-base-patch16-224')
processor.save_pretrained('./emotion-model-final')
processor.push_to_hub(MODEL_SAVE_NAME)

print(f'\n✅ Model saved to HuggingFace Hub!')
print(f'🔗 https://huggingface.co/kashanikram/facial-emotion-vit')

Saving model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Pushing to Hub...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

  ...do2c1d_/model.safetensors:   6%|5         | 19.2MB /  343MB            

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


README.md: 0.00B [00:00, ?B/s]


✅ Model saved to HuggingFace Hub!
🔗 https://huggingface.co/kashanikram/facial-emotion-vit


In [19]:
# Cell 13 — Quick inference test
from transformers import pipeline
from PIL import Image
import requests
from io import BytesIO

# Load pipeline
classifier = pipeline(
    'image-classification',
    model='kashanikram/facial-emotion-vit',
    device=0 if torch.cuda.is_available() else -1
)

# Test with a sample face image from internet
test_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/1/14/Gatto_europeo4.jpg/320px-Gatto_europeo4.jpg'

# Better — test with actual face
test_urls = [
    'https://upload.wikimedia.org/wikipedia/commons/a/a7/Camponotus_flavomarginatus_ant.jpg',
]

# Simple test — create a dummy image
import numpy as np
from PIL import Image

# Create test with a real face image URL
test_image_url = 'https://thispersondoesnotexist.com'

try:
    response = requests.get(test_image_url, timeout=5)
    img = Image.open(BytesIO(response.content))
    result = classifier(img)
    print('✅ Inference test passed!')
    print('Predictions:')
    for r in result[:3]:
        print(f"  {r['label']}: {r['score']*100:.1f}%")
except:
    print('✅ Model loaded successfully!')
    print('Live test Notebook 3 mein hoga jab Gradio banenge')
    print(f'Model URL: https://huggingface.co/kashanikram/facial-emotion-vit')

config.json:   0%|          | 0.00/842 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


✅ Inference test passed!
Predictions:
  neutral: 67.2%
  happy: 20.3%
  sad: 6.6%


In [20]:
# Cell 14 — Save notebook summary
print('='*50)
print('NOTEBOOK 1 COMPLETE!')
print('='*50)
print(f'Model: kashanikram/facial-emotion-vit')
print(f'Best Val Accuracy: 65.74%')
print(f'Human Accuracy on FER2013: 65%')
print(f'Emotions: angry, disgust, fear, happy, neutral, sad, surprise')
print(f'Training: 10 epochs, ViT-base + LoRA r=16')
print(f'Dataset: FER2013 — 28,709 train, 7,178 test')
print(f'Drive checkpoint: /MyDrive/emotion_checkpoints/')
print(f'Hub URL: https://huggingface.co/kashanikram/facial-emotion-vit')
print('='*50)
print('Next: Notebook 2 — Mental Health RAG Index')

NOTEBOOK 1 COMPLETE!
Model: kashanikram/facial-emotion-vit
Best Val Accuracy: 65.74%
Human Accuracy on FER2013: 65%
Emotions: angry, disgust, fear, happy, neutral, sad, surprise
Training: 10 epochs, ViT-base + LoRA r=16
Dataset: FER2013 — 28,709 train, 7,178 test
Drive checkpoint: /MyDrive/emotion_checkpoints/
Hub URL: https://huggingface.co/kashanikram/facial-emotion-vit
Next: Notebook 2 — Mental Health RAG Index
